In [0]:
for t in ["customers", "accounts", "transactions", "branches", "credit_bureau_reports", "payment_gateway_logs"]:
    spark.sql(f"DROP TABLE IF EXISTS neobank.silver.{t}")
spark.sql(f"TRUNCATE TABLE neobank.metadata.pipeline_runs")

In [0]:
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F, Window

catalog = "neobank"
dbutils.widgets.text("run_id", "")
run_id = dbutils.widgets.get("run_id") or datetime.now().strftime("%Y%m%d%H%M%S")


def get_params(table_id):
    rows = (
        spark.table(f"{catalog}.metadata.table_parameters")
        .filter(F.col("table_id") == table_id)
        .collect()
    )
    return {r.param_name: r.param_value for r in rows}


def get_watermark(table_id):
    row = (
        spark.table(f"{catalog}.metadata.table_watermarks")
        .filter(F.col("table_id") == table_id)
        .first()
    )
    return row.last_watermark if row else None


def set_watermark(table_id, value):
    spark.sql(f"""
    MERGE INTO {catalog}.metadata.table_watermarks t
    USING (SELECT {table_id} AS table_id, TIMESTAMP'{value}' AS last_watermark, '{run_id}' AS last_run_id) s
    ON t.table_id = s.table_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *""")


def audit_start(table_id):
    spark.sql(f"DELETE FROM {catalog}.metadata.pipeline_runs "
              f"WHERE run_id = '{run_id}' AND table_id = {table_id}")
    spark.sql(f"""INSERT INTO {catalog}.metadata.pipeline_runs VALUES
              ('{run_id}',
               '{table_id}',
               'silver',
               current_timestamp(),
               NULL,
               'IN_PROGRESS',
               NULL,
               NULL)
               """)


def audit_end(table_id, status, n, err):
    n_sql = "NULL" if n is None else n
    e_sql = "NULL" if err is None else "'" + str(err)[:500].replace("'", "''") + "'"
    spark.sql(f"""UPDATE {catalog}.metadata.pipeline_runs
              SET end_time = current_timestamp(),
                  status = '{status}',
                  records_processed = {n_sql},
                  error_message = {e_sql}
              WHERE run_id = '{run_id}'
               AND table_id = {table_id}""")

In [0]:
TRANSFORMS = {
    "branches": lambda df: (
        df.filter("branch_code IS NOT NULL")
        .withColumn("branch_code", F.trim("branch_code"))
        .withColumn("branch_name", F.trim("branch_name"))
        .drop("created_at")
    ),
    "customers": lambda df: (
        df.filter("customer_id IS NOT NULL")
        .withColumn("first_name", F.initcap(F.trim("first_name")))
        .withColumn("last_name", F.initcap(F.trim("last_name")))
        .withColumn("email", F.lower(F.trim("email")))
        .withColumn("pan_number", F.upper(F.trim("pan_number")))
        .drop("created_at", "updated_at")
    ),
    "accounts": lambda df: (
        df.filter("account_id IS NOT NULL")
        .withColumn("account_type", F.upper(F.trim("account_type")))
        .drop("created_at", "updated_at")
    ),
    "transactions": lambda df: (
        df.filter("txn_id IS NOT NULL")
        .withColumn("txn_type", F.upper(F.trim("txn_type")))
        .drop("created_at")
    ),
    "payment_gateway_logs": lambda df: df.filter("txn_id IS NOT NULL"),
    "credit_bureau_reports": lambda df: df.filter("customer_id IS NOT NULL"),
}

In [0]:
def process_table(r):
    tid, name = r.table_id, r.table_name
    p = get_params(tid)
    load_type = p["loadtype"]
    wm_col = p.get("watermark_column")
    keys = [k.strip() for k in p.get("primary_key", "").split(",") if k.strip()]
    target = f"{catalog}.{r.silver_schema}.{name}"

    audit_start(tid)
    n, status, err = None, "SUCCESS", None
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{r.silver_schema}")
        src = spark.table(f"{catalog}.{r.bronze_schema}.{name}")

        new_wm = None
        if load_type in ("merge", "append"):
            src = src.filter(F.col(wm_col) > F.lit(get_watermark(tid)))
            new_wm = src.agg(F.max(wm_col)).first()[0]

        df = TRANSFORMS[name](src)

        if load_type == "merge":
            w = Window.partitionBy(*keys).orderBy(F.col("insert_timestamp").desc())
            df = (
                df.withColumn("_rn", F.row_number().over(w))
                .filter("_rn = 1")
                .drop("_rn")
            )

        df = df.drop("insert_timestamp", "_rescued_data").withColumn(
            "update_timestamp", F.current_timestamp()
        )
        n = df.count()

        if load_type == "full":
            df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
                target
            )
        elif n > 0:
            if load_type == "append" or not spark.catalog.tableExists(target):
                df.write.mode("append").saveAsTable(target)
            elif load_type == "merge":
                cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)
                (
                    DeltaTable.forName(spark, target)
                    .alias("t")
                    .merge(df.alias("s"), cond)
                    .whenMatchedUpdateAll()
                    .whenNotMatchedInsertAll()
                    .execute()
                )
            else:
                raise ValueError(f"Unknown load type: {load_type}")

        if new_wm is not None:
            set_watermark(tid, new_wm)
    except Exception as e:
        status, err = "Failed", e
    finally:
        audit_end(tid, status, n, err)
    return status, err

In [0]:
run_id = dbutils.widgets.get("run_id") or datetime.now().strftime("%Y%m%d%H%M%S")
print("run_id:", run_id)

rows = (
    spark.table(f"{catalog}.metadata.tables")
    .filter("source_system = 'blob' AND active_flag = true")
    .orderBy("load_order")
    .collect()
)

failed = []
for r in rows:
    status, err = process_table(r)
    print(f"{r.table_name}: {status}", "" if err is None else f"-> {err}")
    if status == "FAILED": failed.append(r.table_name)

if failed:
    raise Exception(f"Failed tables: {failed}")

In [0]:
for t in ["customers","accounts","transactions","branches","credit_bureau_reports","payment_gateway_logs"]:
    print(t, spark.table(f"neobank.silver.{t}").count())
display(spark.table("neobank.metadata.pipeline_runs"))
display(spark.table("neobank.metadata.table_watermarks"))

In [0]:
# for t in ["customers","accounts","transactions","branches","credit_bureau_reports","payment_gateway_logs"]:
#     spark.sql(f"DROP TABLE IF EXISTS neobank.silver.{t}")
# spark.sql("TRUNCATE TABLE neobank.metadata.pipeline_runs")
# spark.sql("""UPDATE neobank.metadata.table_watermarks
#              SET last_watermark = TIMESTAMP'1900-01-01 00:00:00', last_run_id = NULL""")